In [ ]:
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from tvbwidgets.ui.head_widget import HeadWidget
from tvbwidgets.ui.connectivity_matrix_editor_widget import ConnectivityMatrixEditor
from tvb.datatypes.connectivity import Connectivity
import k3d


conn = Connectivity().from_file()
conn.configure()


class CustomMatrixEditor(widgets.VBox, ConnectivityMatrixEditor):
   

    def __init__(self, connectivity, viewer, **kwargs):
        ConnectivityMatrixEditor.__init__(self, connectivity, **kwargs)
        widgets.VBox.__init__(self)
        self.viewer = viewer
        self.output = widgets.Output()

        with self.output:
            self.display()

        self.children = [self.output]

    def on_cell_clicked(self, x, y, matrix_name):
        
        super().on_cell_clicked(x, y, matrix_name)

       
        actual_row = int(self.from_row + self.row)
        actual_col = int(self.from_col + self.col)

       
        centres = self.connectivity.centres
        selected = []

        if 0 <= actual_row < len(centres):
            selected.append(centres[actual_row].tolist())
        if 0 <= actual_col < len(centres) and actual_col != actual_row:
            selected.append(centres[actual_col].tolist())

        if selected and hasattr(self.viewer, "highlight_centres"):
            self.viewer.highlight_centres(selected)


class CustomHeadViewer(HeadWidget):
   

    def __init__(self, connectivity, **kwargs):
        super().__init__([connectivity], **kwargs)
        self._connectivity  = connectivity
        self._highlight_obj = None

    def highlight_centres(self, selected_centres):
        
        if not selected_centres:
            return

       
        if self._highlight_obj is not None:
            try:
                self.plot -= self._highlight_obj
            except Exception:
                pass
            self._highlight_obj = None

        
        points = np.array(selected_centres, dtype=np.float32)
        self._highlight_obj = k3d.points(
            positions=points,
            point_size=14,      
            shader='dot',
            color=0xff0000,     
            name='SelectedRegions',
        )
        self.plot += self._highlight_obj




custom_viewer = CustomHeadViewer(conn, width=700, height=550)
custom_matrix_editor = CustomMatrixEditor(conn, viewer=custom_viewer)



side_by_side = widgets.HBox(
    [custom_matrix_editor, custom_viewer],
    layout=widgets.Layout(
        width='100%',
        justify_content='space-between',
        padding='10px',
        border='1px solid #eee',
    )
)

display(side_by_side)

22-03-2026 04:24:36 - INFO - tvbwidgets - Version: 2.3.2
2026-03-22 16:24:39,135 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.


C:\Users\deept\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\traittypes\traittypes.py:98: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
